<!-- nav -->
[← 7 · Generating test data](07_Generating_Test_Data.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)

# 8 · Rewriting step by step

**The problem.** Generating code from a model, migrating data, tidying a catalogue: each is a sequence of rewrites,
and many of them decide something on the way, which case to write a label in, what size to give an item. A script
decides silently, and next time perhaps differently. With people and agents in the loop, each decision must be
visible, made by whoever should make it, and recorded, so that a rerun makes it the same way.

A **transform** is a rewrite with a *before* and an *after*, both predicates over the same symbols, and parameters for
what it decides. A **session** applies transforms to a store one decision at a time, and records every step as data.
This case study labels and sizes the items on a shelf; its small schemas are defined here, in the first cell.

In [1]:
from mbse.Expressions import Expressions as E
from mbse.Patterns import Predicates, Queries, Transforms
from mbse.Schemas.Framework import JSON, Proxies, Schemas, Stores

Held = Schemas.OfRelation.Builder().name("Held").links("shelf", "item").create()
Shelf = (Schemas.OfObject.Builder().name("Shelf").ref().singleton("Shelf")
         .relations(lambda r: r.name("items").of(Held).me("shelf")).create())
Item = (Schemas.OfObject.Builder().name("Item").ref()
        .properties(lambda p: p.name("name").of(lambda t: t.as_native(str)),
                    lambda p: p.name("label").of(lambda t: t.as_native(str)),
                    lambda p: p.name("size").of(lambda t: t.as_native(int)))
        .relations(lambda r: r.name("shelves").of(Held).me("item")).create())


def shelf(*names):
    """A store whose shelf holds an item of each name, in order: its data is what the shelf reaches."""
    store = Proxies.OfStore()
    for schema in (Shelf, Item, Held):
        store.register(schema)
    for name in names:
        item = store.Item().name(name).create()
        store.Shelf(store.singleton("Shelf")).items(lambda e, item=item: e.item(item)).update()
    return store


i = E.variable("i")
Named = Predicates.OfPredicate.Builder().name("Named").symbols({"i": Item}).requires(i.has("name")).create()
Labelled = Predicates.OfPredicate.Builder().name("Labelled").symbols({"i": Item}).requires(i.has("label")).create()
Sized = Predicates.OfPredicate.Builder().name("Sized").symbols({"i": Item}).requires(i.has("size")).create()

<a id="T08.1"></a>
## 08.1 · a transform

A transform has a name, a before and an after, the **parameters** it decides, and a **rewrite** that makes its after
hold. `Label` writes an item's name as its label, in upper or lower case: `case` is a parameter whose schema is a union
of two options, so it has two values. `Size` gives a labelled item a size, an int, which has too many values to list:

In [2]:
Option = Schemas.OfObject.Builder().create()
Case = Schemas.OfUnion.Builder().branches(lambda b: b.name("upper").of(Option), lambda b: b.name("lower").of(Option)).create()


def label(store, match, arguments):
    item = match["i"]
    store.Item(item).label(item.name.upper() if arguments["case"] == "upper" else item.name.lower()).update()


def size(store, match, arguments):
    store.Item(match["i"]).size(arguments["size"]).update()


Label = Transforms.Transform("Label", Named, Labelled, [lambda p: p.name("case").of(Case)], label)
Size = Transforms.Transform("Size", Labelled, Sized, [lambda p: p.name("size").of(lambda t: t.as_native(int))], size)
Label.check(), Size.check(), list(Label.parameters), list(Size.parameters)

([], [], ['case'], ['size'])

`check()` finds nothing wrong: the before and after have the same symbols, and each transform has a rewrite.

<a id="T08.2"></a>
## 08.2 · candidates

A session offers **candidates**: a transform at a match where its before holds and its after doesn't. Each value of a
finite parameter is a candidate of its own, so bolt and nut each have two; `Size` has none yet, since nothing is
labelled:

In [3]:
store = shelf("bolt", "nut")
session = Transforms.Session(store, [Label, Size])
[(c.transform.name, c.match["i"].name, dict(c.arguments)) for c in session.candidates()]

[('Label', 'bolt', {'case': 'upper'}),
 ('Label', 'bolt', {'case': 'lower'}),
 ('Label', 'nut', {'case': 'upper'}),
 ('Label', 'nut', {'case': 'lower'})]

A candidate decides everything but its **open** parameters, those with no finite domain, which whoever takes it must
answer.

<a id="T08.3"></a>
## 08.3 · taking a step yourself

The caller decides: `take` applies one candidate, one decision. The data changes, and so do the candidates: bolt's
other case is gone, and its size is enabled, open until you answer it:

In [4]:
session.take(session.candidates()[0])  # bolt, upper
sizing = session.candidates()[2]
print([(c.transform.name, c.match["i"].name, dict(c.arguments)) for c in session.candidates()], sizing.open)
session.take(sizing.answer(size=3))
sorted((item.name, item.label, item.size) for item in store.extent("Item"))

[('Label', 'nut', {'case': 'upper'}), ('Label', 'nut', {'case': 'lower'}), ('Size', 'bolt', {})] ('size',)


[('bolt', 'BOLT', 3), ('nut', None, None)]

<a id="T08.4"></a>
## 08.4 · a rewrite must establish its after

A session checks each step: after the rewrite, the after must hold. A rewrite that doesn't do what its transform says
is refused, by name and match:

In [5]:
Forgetful = Transforms.Transform("Forgetful", Named, Labelled, rewrite=lambda store, match, arguments: None)
forgetting = Transforms.Session(shelf("bolt"), [Forgetful])
try:
    forgetting.take(forgetting.candidates()[0])
except ValueError as error:
    print(error)

'Forgetful' did not establish its after at i=Shelf/items[0]


<a id="T08.5"></a>
## 08.5 · a policy decides

Where you'd rather not decide each step, a **policy** does. Its clauses weigh candidates by transform and arguments,
and answer open parameters; `candidates(policy)` ranks them, a recommendation:

In [6]:
policy = Transforms.Policy(Transforms.Clause("Label", {"case": "lower"}, 2.0), Transforms.Clause("Size", {"size": 5}))
store = shelf("bolt", "nut")
session = Transforms.Session(store, [Label, Size])
[(c.transform.name, c.match["i"].name, dict(c.arguments), c.score) for c in session.candidates(policy)]

[('Label', 'bolt', {'case': 'lower'}, 2.0),
 ('Label', 'nut', {'case': 'lower'}, 2.0),
 ('Label', 'bolt', {'case': 'upper'}, 0),
 ('Label', 'nut', {'case': 'upper'}, 0)]

`step_over(policy)` takes the policy's first candidate, and `run(policy)` steps until the policy has nothing it can
take; a candidate no clause weighs is never the policy's to take:

In [7]:
print(session.step_over(policy), session.run(policy), session.done, [step.by for step in session.steps])
sorted((item.name, item.label, item.size) for item in store.extent("Item"))

True 3 True ['policy', 'policy', 'policy', 'policy']


[('bolt', 'bolt', 5), ('nut', 'nut', 5)]

<a id="T08.6"></a>
## 08.6 · composite transforms, stepped into or over

A transform with **parts** instead of a rewrite is composite: `Finish` is `Label`, then `Size`. Step into it, and you
decide inside it, for its match alone; step over it, and the policy does:

In [8]:
Finished = (Predicates.OfPredicate.Builder().name("Finished").symbols({"i": Item})
            .requires(i.has("label").and_(i.has("size"))).create())
Finish = Transforms.Transform("Finish", Named, Finished, parts=[Label, Size])
session = Transforms.Session(shelf("bolt", "nut"), [Finish])
inner = session.step_in(session.candidates()[0])  # bolt: the caller decides inside
inner.take(inner.candidates()[1])  # lower
inner.take(inner.candidates()[0].answer(size=2))
session.step_over(Transforms.Policy(Transforms.Clause("Finish"), Transforms.Clause("Label", {"case": "upper"}),
                                    Transforms.Clause("Size", {"size": 4})))  # nut: the policy decides
[(s.transform, s.by, s.mode, [(x.transform, x.by, dict(x.arguments)) for x in s.steps]) for s in session.steps]

[('Finish',
  'caller',
  'in',
  [('Label', 'caller', {'case': 'lower'}), ('Size', 'caller', {'size': 2})]),
 ('Finish',
  'policy',
  'over',
  [('Label', 'policy', {'case': 'upper'}), ('Size', 'policy', {'size': 4})])]

Each step records who decided it, `caller` or `policy`, and a composite's steps hold its parts' steps.

<a id="T08.7"></a>
## 08.7 · the trace is data

The steps taken are a **trace**, written as data like any other, and read back as steps. Each step's **key** names its
transform and where it matched, by path from the store's roots:

In [9]:
traces = Transforms.register(Proxies.OfStore())  # a store of the trace's schemas
text = JSON.ToJSON(traces)(Transforms.Trace, session.trace(traces))
print(text[:300], "...")
[step.key for step in Transforms.steps(traces, session.trace(traces))]

{"root": "s0", "objects": {"s0": {"steps": [{"transform": "Finish", "match": [{"symbol": "i", "element": "Shelf/items[0]"}], "by": "caller", "mode": "in", "steps": [{"transform": "Label", "match": [{"symbol": "i", "element": "Shelf/items[0]"}], "arguments": [{"name": "case", "value": {"str": "lower" ...


['Finish(i=Shelf/items[0])', 'Finish(i=Shelf/items[1])']

<a id="T08.8"></a>
## 08.8 · a rerun reuses decisions

A rerun given the earlier steps takes each earlier decision again where its key still occurs, here on a store with
one more item. `run()` with no policy takes only those; the new item's steps are still to decide:

In [10]:
decide = Transforms.Policy(Transforms.Clause("Label", {"case": "upper"}), Transforms.Clause("Size", {"size": 2}))
first = Transforms.Session(shelf("a", "b"), [Label, Size])
first.run(decide)
again = Transforms.Session(shelf("a", "b", "c"), [Label, Size], earlier=first.steps)
again.run(), [step.by for step in again.steps], again.done, [(c.transform.name, c.match["i"].name) for c in again.candidates()][:1]

(4, ['reused', 'reused', 'reused', 'reused'], False, [('Label', 'c')])

A decision whose key no longer occurs is an **orphan**, reported rather than dropped, and `diff` compares two runs by
key: the steps only the later took (here, the new item's), those only the earlier took, and those both took with other
arguments (here, a run that chose lower case):

In [11]:
again.run(Transforms.Policy(Transforms.Clause("Label", {"case": "lower"}), Transforms.Clause("Size", {"size": 3})))
fewer = Transforms.Session(shelf("a"), [Label, Size], earlier=first.steps)
fewer.run()
lower = Transforms.Session(shelf("a", "b"), [Label, Size])
lower.run(Transforms.Policy(Transforms.Clause("Label", {"case": "lower"}), Transforms.Clause("Size", {"size": 2})))
print([s.key for s in fewer.orphans])
print([s.key for s in Transforms.diff(first.steps, again.steps).added])
[(a.key, dict(a.arguments), dict(b.arguments)) for a, b in Transforms.diff(first.steps, lower.steps).changed]

['Label(i=Shelf/items[1])', 'Size(i=Shelf/items[1])']
['Label(i=Shelf/items[2])', 'Size(i=Shelf/items[2])']


[('Label(i=Shelf/items[0])', {'case': 'upper'}, {'case': 'lower'}),
 ('Label(i=Shelf/items[1])', {'case': 'upper'}, {'case': 'lower'})]

<a id="T08.9"></a>
## 08.9 · what each step wrote

A rewrite may say what it wrote, by role; the step links what it matched and what it wrote, so you can ask which step
made something. `Pack` boxes each item: a new box on the shelf, labelled with the item's name:

In [12]:
def pack(store, match, arguments):
    """Puts a box for the item on the shelf, labels the item packed, and says what it wrote, by role."""
    item = match["i"]
    box = store.Item().name("box").label(item.name).create()
    store.Shelf(store.singleton("Shelf")).items(lambda e: e.item(box)).update()
    store.Item(item).label("packed").update()
    return {"box": box, "item": item}


Unlabelled = (Predicates.OfPredicate.Builder().name("Unlabelled").symbols({"i": Item})
              .requires(i.has("name")).forbids(i.has("label")).create())
Pack = Transforms.Transform("Pack", Unlabelled, Labelled, rewrite=pack)
packing = Transforms.Session(shelf("bolt", "nut"), [Pack])
packing.run(Transforms.Policy(Transforms.Clause("Pack")))
bolts = packing.steps[0]
box = bolts.wrote[0].element
[link.role for link in bolts.wrote], box.label, packing.wrote(box) is bolts

(['box', 'item'], 'bolt', True)

In a trace, each link is a path. Steps are objects too: combined with the session's store, a predicate queries them
like any other data, here each box and the step that made it (`Stores.Combined` is mbse-schemas 0.10.0's, which this
release pins):

In [13]:
print([(link.role, link.path) for link in Transforms.steps(traces, packing.trace(traces))[0].wrote])
combined = Stores.Combined(packing.store, Transforms.records(packing))
st, b = E.variable("st"), E.variable("b")
Boxed = (Predicates.OfPredicate.Builder().name("Boxed").symbols({"st": Transforms.StepObject, "b": Item})
         .requires(Predicates.Contains(st.wrote, lambda e: e.element == b and e.role == "box")).create())
[(m["st"].key, m["b"].label) for m in Queries.select(combined, Boxed)]

[('box', 'Shelf/items[0]'), ('item', 'Shelf/items[1]')]


[('Pack(i=Shelf/items[0])', 'bolt'), ('Pack(i=Shelf/items[1])', 'nut')]

## Why it works this way

- **Every decision is a step.** A rewrite decides only what its parameters say, so each decision is visible, made by the
  caller or delegated to a policy, and recorded: [CAP-074: transforms applied step by step](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-074.md); [PRI-014: every ambiguity a decision](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-014).
- **Before and after are predicates.** A transform is enabled where its before holds and its after doesn't, so a step
  that is done is never offered again, and a rewrite that fails its after is refused.
- **Links, not copies.** A step links what it matched and wrote, so what a step made can be traced back to it:
  [CAP-075: steps linked to what they matched and wrote](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-075.md).
- **Keys make reruns cheap.** A step's key is its transform and its match's paths, so a rerun on changed data takes
  every decision that still applies and reports the rest: [CAP-076: reruns reuse decisions](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-076.md).

<!-- nav -->
[← 7 · Generating test data](07_Generating_Test_Data.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)